# Baseline Modeling - FINAL VERSION

## Configuration:
- **Daily**: SARIMA, Test = 4 weeks, Linear Interpolation
- **Weekly**: THETA, Test = 52 weeks, Smart Gap Filling

## Gap Strategy:
- **< 5% gaps**: Linear Interpolation (realistic)
- **≥ 5% gaps**: Dropna (too many gaps)
- **No false zeros, no plateaus**

In [1]:
# IMPORTS
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, r2_score
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive, Theta

from Favorita_TSA.models.data_preparation import build_dataframes
from Favorita_TSA.viz.color_manager import ColorManager
from Favorita_TSA.viz.ploty_theme import set_plotly_theme

set_plotly_theme()
colors = ColorManager().get_colors()
print("✅ Imports loaded")

/Users/kiko/Desktop/github/Group-Work-Favorita-Forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports loaded


In [2]:
# CONFIGURATION

ITEMS_TO_MODEL = {
    "daily_smooth": {"store": 25, "item": 115611},
    "daily_erratic": {"store": 44, "item": 103520},
    "weekly_smooth": {"store": 24, "item": 1503844},
    "weekly_erratic": {"store": 51, "item": 1239986},
}

TEST_WEEKS_DAILY = 4
TEST_WEEKS_WEEKLY = 52
GAP_THRESHOLD = 0.05  # 5%
TEMPLATE = "plotly_white"

print("✅ Configuration loaded")
print(f"   Daily:  SARIMA, Test = {TEST_WEEKS_DAILY} weeks")
print(f"   Weekly: THETA, Test = {TEST_WEEKS_WEEKLY} weeks")
print(
    f"   Gap Strategy: Linear (<{GAP_THRESHOLD*100}%) | Dropna (≥{GAP_THRESHOLD*100}%)"
)

✅ Configuration loaded
   Daily:  SARIMA, Test = 4 weeks
   Weekly: THETA, Test = 52 weeks
   Gap Strategy: Linear (<5.0%) | Dropna (≥5.0%)


In [3]:
# HELPER FUNCTIONS


def load_and_prepare(df, store, item, freq="D"):
    """
    Load and prepare time series.

    WICHTIG: Für aggregierte Weekly Daten ist df bereits gefiltert!
    """

    # Check ob df bereits gefiltert ist (von aggregate_to_weekly)
    if len(df) < 1000 and "week_start" in df.columns:
        # Bereits gefiltert und aggregiert
        ts = df.copy()
        date_col = "week_start"
        target_col = "unit_sales" if "unit_sales" in ts.columns else "target_sales"

        print("   Using pre-filtered data")
    else:
        # Normale Filterung
        if freq == "W":
            date_col = (
                "week_start"
                if "week_start" in df.columns
                else df.reset_index().columns[0]
            )
            if date_col != "week_start":
                df = df.reset_index()
                if "week_start" not in df.columns:
                    raise ValueError("Weekly needs 'week_start'")
                date_col = "week_start"
        else:
            date_col = "date" if "date" in df.columns else df.reset_index().columns[0]
            if date_col != "date":
                df = df.reset_index()
                if "date" not in df.columns:
                    raise ValueError("Daily needs 'date'")
                date_col = "date"

        print(f"   Using: '{date_col}' (freq={freq})")

        # Filter
        ts = df[(df["store_nbr"] == store) & (df["item_nbr"] == item)].copy()
        if len(ts) == 0:
            raise ValueError(f"No data for Store {store}, Item {item}")

        target_col = "unit_sales" if "unit_sales" in ts.columns else "target_sales"

    # Format
    ts[date_col] = pd.to_datetime(ts[date_col])
    ts = ts.sort_values(date_col)
    ts = ts[[date_col, target_col]].rename(columns={date_col: "ds", target_col: "y"})
    ts["unique_id"] = f"store_{store}_item_{item}"

    print(f"📊 Loaded: {len(ts)} obs | {ts['ds'].min()} to {ts['ds'].max()}")
    print(f"   Mean: {ts['y'].mean():.2f}, Std: {ts['y'].std():.2f}")

    # Remove NaN
    n_nan = ts["y"].isna().sum()
    if n_nan > 0:
        print(f"  ⚠️  Removing {n_nan} NaN values")
        ts = ts.dropna(subset=["y"])

    # Gap handling
    if freq == "D":
        # Daily: Create continuous series
        if not ts["ds"].duplicated().any():
            full_range = pd.date_range(ts["ds"].min(), ts["ds"].max(), freq="D")
            ts_cont = ts.set_index("ds").reindex(full_range).reset_index()
            ts_cont.columns = ["ds" if c == "index" else c for c in ts_cont.columns]

            n_filled = ts_cont["y"].isna().sum()
            if n_filled > 0:
                ts_cont["y"] = ts_cont["y"].interpolate(method="linear")
                print(f"  ✅ Filled {n_filled} daily gaps with interpolation")

                if "unique_id" in ts_cont.columns:
                    ts_cont["unique_id"] = (
                        ts_cont["unique_id"]
                        .fillna(method="ffill")
                        .fillna(method="bfill")
                    )

                ts = ts_cont
    else:
        # Weekly: NO gap filling needed for properly aggregated data
        print("  ✅ Weekly aggregated data ready (no gap filling needed)")

    # Final safety
    if ts["y"].isna().any():
        print(f"  ⚠️  SAFETY: Filling {ts['y'].isna().sum()} NaN with 0")
        ts["y"] = ts["y"].fillna(0)

    print(f"   Final: {len(ts)} observations")

    return ts


def train_test_split(df, test_weeks=4):
    """Time-based split."""
    cutoff = df["ds"].max() - pd.Timedelta(weeks=test_weeks)

    train = df[df["ds"] <= cutoff].copy()
    test = df[df["ds"] > cutoff].copy()

    print(f"✂️  Train: {len(train)} obs | Test: {len(test)} obs")
    print(f"   Train: {train['ds'].min()} to {train['ds'].max()}")
    print(f"   Test:  {test['ds'].min()} to {test['ds'].max()}")

    return train, test


print("✅ Helper functions loaded")

✅ Helper functions loaded


In [4]:
# DATEN LADEN

PROJECT_ROOT = Path("..").resolve()
os.chdir(f"{PROJECT_ROOT}")

dfs = build_dataframes()

daily_smooth = dfs["smooth_daily"]  # Smooth  aus daily  Matrix
daily_erratic = dfs["erratic_daily"]  # Erratic aus daily  Matrix
weekly_smooth = dfs["smooth_weekly"]  # Smooth  aus weekly Matrix
weekly_erratic = dfs["erratic_weekly"]  # Erratic aus weekly Matrix

print("📂 Lade Daten...")


print(f"✅ Daily Smooth:   {daily_smooth.shape}")
print(f"✅ Daily Erratic:  {daily_erratic.shape}")
print(f"✅ Weekly Smooth:  {weekly_smooth.shape}")
print(f"✅ Weekly Erratic: {weekly_erratic.shape}")

Loading fact table …
Loading forecastability matrices …
Building DataFrames …
  smooth_daily         36,560 store-item pairs     39,523,827 rows
  erratic_daily        35,933 store-item pairs     36,705,401 rows
  smooth_weekly        47,196 store-item pairs     30,042,814 rows
  erratic_weekly       19,734 store-item pairs     10,182,367 rows
📂 Lade Daten...
✅ Daily Smooth:   (39523827, 12)
✅ Daily Erratic:  (36705401, 12)
✅ Weekly Smooth:  (30042814, 12)
✅ Weekly Erratic: (10182367, 12)


In [5]:
import mlflow

# Assumption: you already have PROJECT_ROOT like in your notebook
MLRUNS_DIR = PROJECT_ROOT / "mlruns"

# Local file based tracking inside the repo
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR.as_posix()}")

# One experiment for your baseline iteration history
mlflow.set_experiment("favorita_baseline_store_item")
print(f"✅ MLflow configured with tracking URI: {mlflow.get_tracking_uri()}")

✅ MLflow configured with tracking URI: file:///Users/kiko/Desktop/github/Group-Work-Favorita-Forecasting/mlruns


/Users/kiko/Desktop/github/Group-Work-Favorita-Forecasting/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [6]:
# =============================================================================
# NEUE ZELLE: AGGREGATE TO TRUE WEEKLY
# =============================================================================

print("📊 Aggregating daily data to true weekly...")


def aggregate_to_weekly(df, store, item):
    """
    Aggregiert daily data zu weekly.

    Parameters
    ----------
    df : DataFrame
        Daily data mit 'date' oder 'week_start' column
    store : int
        Store number
    item : int
        Item number

    Returns
    -------
    DataFrame
        Weekly aggregated data mit 'week_start', 'unit_sales', etc.
    """
    # Filter für Store-Item
    ts = df[(df["store_nbr"] == store) & (df["item_nbr"] == item)].copy()

    if len(ts) == 0:
        print(f"  ⚠️  No data for Store {store}, Item {item}")
        return None

    # Determine date column
    if "date" in ts.columns:
        date_col = "date"
    elif "week_start" in ts.columns:
        date_col = "week_start"
    else:
        ts = ts.reset_index()
        if "date" in ts.columns:
            date_col = "date"
        elif "week_start" in ts.columns:
            date_col = "week_start"
        else:
            date_col = ts.columns[0]

    print(f"  Using: {date_col}")

    # Convert to datetime
    ts[date_col] = pd.to_datetime(ts[date_col])

    # Check current frequency
    ts = ts.sort_values(date_col)
    time_diffs = ts[date_col].diff().dt.days.dropna()
    median_diff = time_diffs.median()

    print(f"  Current frequency: {median_diff:.1f} days between observations")

    if median_diff >= 6:
        print(" Already weekly-ish, but will aggregate anyway for consistency")

    # Create week identifier
    ts["week_start"] = ts[date_col].dt.to_period("W").dt.start_time

    # Determine target column
    target_col = "unit_sales" if "unit_sales" in ts.columns else "target_sales"

    # Aggregate to weekly
    ts_weekly = (
        ts.groupby("week_start")
        .agg(
            {
                target_col: "sum",  # Summe der Verkäufe pro Woche
                "store_nbr": "first",
                "item_nbr": "first",
            }
        )
        .reset_index()
    )

    print(f"  ✅ Aggregated: {len(ts)} daily obs → {len(ts_weekly)} weekly obs")
    print(
        f"     Period: {ts_weekly['week_start'].min()} to {ts_weekly['week_start'].max()}"
    )

    return ts_weekly


# Aggregiere Weekly Smooth
print("\n1. Weekly Smooth:")
weekly_smooth_agg = aggregate_to_weekly(
    weekly_smooth,
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
)

# Aggregiere Weekly Erratic
print("\n2. Weekly Erratic:")
weekly_erratic_agg = aggregate_to_weekly(
    weekly_erratic,
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
)

print("\n✅ Weekly aggregation complete!")
print(
    f"   Weekly Smooth:  {weekly_smooth_agg.shape if weekly_smooth_agg is not None else 'None'}"
)
print(
    f"   Weekly Erratic: {weekly_erratic_agg.shape if weekly_erratic_agg is not None else 'None'}"
)

📊 Aggregating daily data to true weekly...

1. Weekly Smooth:
  Using: date
  Current frequency: 1.0 days between observations
  ✅ Aggregated: 993 daily obs → 147 weekly obs
     Period: 2013-12-30 00:00:00 to 2017-08-14 00:00:00

2. Weekly Erratic:
  Using: date
  Current frequency: 7.0 days between observations
 Already weekly-ish, but will aggregate anyway for consistency
  ✅ Aggregated: 184 daily obs → 179 weekly obs
     Period: 2013-11-04 00:00:00 to 2017-07-31 00:00:00

✅ Weekly aggregation complete!
   Weekly Smooth:  (147, 4)
   Weekly Erratic: (179, 4)


In [ ]:
# BASELINE FUNCTION


def run_baseline_plotly(
    df,
    pattern,
    store,
    item,
    freq="D",
    season_length=7,
    test_weeks=4,
    model_type="sarima",
):
    """Run baseline with proper merge handling."""

    print(f"\n{'='*70}\n🎯 {pattern.upper()} | {model_type.upper()}\n{'='*70}")

    run_name = f"{pattern}_store{store}_item{item}_season{season_length}_freq{freq}_model{model_type}"

    with mlflow.start_run(run_name=run_name):
        # MLflow parameters
        mlflow.log_params(
            {
                "pattern": pattern,
                "store": store,
                "item": item,
                "freq": freq,
                "season_length": season_length,
                "test_weeks": test_weeks,
                "model_type": model_type,
            }
        )

        # 1. Prepare
        ts = load_and_prepare(df, store, item, freq=freq)

        # 2. Split
        train, test = train_test_split(ts, test_weeks=test_weeks)

        # 3. Train primary model
        if model_type == "theta":
            print(f"\n🤖 Training Theta (season={season_length})...")
            model_primary = StatsForecast(
                models=[Theta(season_length=season_length)], freq=freq, n_jobs=1
            )
            primary_col = "Theta"
        else:
            print(f"\n🤖 Training SARIMA (season={season_length})...")
            model_primary = StatsForecast(
                models=[AutoARIMA(season_length=season_length)], freq=freq, n_jobs=1
            )
            primary_col = "AutoARIMA"

        model_primary.fit(train)

        # 4. Forecast
        horizon = len(test)
        print(f"   Forecasting {horizon} periods...")
        forecasts_primary = model_primary.predict(h=horizon)

        # DEBUG: Check forecasts
        print(f"\n   DEBUG: Forecasts shape: {forecasts_primary.shape}")
        print(f"   DEBUG: Forecasts columns: {forecasts_primary.columns.tolist()}")
        print(f"   DEBUG: Test shape: {test.shape}")

        # Reset index für sauberes Merge
        forecasts_reset = forecasts_primary.reset_index()
        print(f"   DEBUG: Forecasts reset columns: {forecasts_reset.columns.tolist()}")
        print(
            f"   DEBUG: First forecast dates: {forecasts_reset['ds'].head(3).tolist()}"
        )
        print(f"   DEBUG: First test dates: {test['ds'].head(3).tolist()}")

        # Merge
        test_primary = test.copy()
        test_primary[primary_col] = forecasts_reset[primary_col].values

        print(
            f"   DEBUG: After merge, NaN count: {test_primary[primary_col].isna().sum()}"
        )

        # Evaluate
        actuals = test_primary["y"].values
        preds_primary = test_primary[primary_col].values
        mask = ~np.isnan(preds_primary) & ~np.isnan(actuals)

        print(f"   DEBUG: Mask sum: {mask.sum()} / {len(mask)}")

        if mask.sum() == 0:
            print("\n❌ ERROR DETAILS:")
            print(f"   Actuals NaN: {np.isnan(actuals).sum()}")
            print(f"   Predictions NaN: {np.isnan(preds_primary).sum()}")
            print(f"   Test dates: {test['ds'].min()} to {test['ds'].max()}")
            print(
                f"   Forecast dates: {forecasts_reset['ds'].min()} to {forecasts_reset['ds'].max()}"
            )
            raise ValueError("All predictions are NaN!")

        mae_primary = mean_absolute_error(actuals[mask], preds_primary[mask])
        r2_primary = r2_score(actuals[mask], preds_primary[mask])
        print(f"   ✅ MAE: {mae_primary:.2f} | R²: {r2_primary:.3f}")

        # 5. Train naive
        print("\n🤖 Training Naive...")
        model_naive = StatsForecast(
            models=[SeasonalNaive(season_length=season_length)], freq=freq, n_jobs=1
        )
        model_naive.fit(train)
        forecasts_naive = model_naive.predict(h=horizon)

        test_naive = test.copy()
        test_naive["SeasonalNaive"] = forecasts_naive.reset_index()[
            "SeasonalNaive"
        ].values

        preds_naive = test_naive["SeasonalNaive"].values
        mae_naive = mean_absolute_error(actuals[mask], preds_naive[mask])
        r2_naive = r2_score(actuals[mask], preds_naive[mask])
        print(f"   ✅ MAE: {mae_naive:.2f} | R²: {r2_naive:.3f}")

        improvement = (mae_naive - mae_primary) / mae_naive * 100
        print(f"\n📊 Improvement: {improvement:+.1f}%")

        # COLOR CONFIGURATIONs

        COL_TRAIN = colors.blue_brand
        COL_ACTUAL = colors.blue_brand
        COL_SARIMA = colors.red_brand
        COL_NAIVE = colors.gold_brand

        # 6. Plots
        model_name = model_type.upper()
        fig = make_subplots(
            rows=2,
            cols=1,
            subplot_titles=(
                f"{pattern.upper()} - {model_name} vs Naive",
                f"Test Period (Improvement: {improvement:+.1f}%)",
            ),
            vertical_spacing=0.12,
        )

        fig.add_trace(
            go.Scatter(
                x=train["ds"],
                y=train["y"],
                mode="lines",
                name="Train",
                line={"color": COL_TRAIN, "width": 1},
                opacity=0.6,
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_primary["ds"],
                y=test_primary["y"],
                mode="lines+markers",
                name="Actual",
                line={"color": COL_ACTUAL, "width": 2},
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_primary["ds"],
                y=test_primary[primary_col],
                mode="lines+markers",
                name=f"{model_name} (MAE={mae_primary:.1f})",
                line={"color": COL_SARIMA, "width": 2, "dash": "dash"},
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_naive["ds"],
                y=test_naive["SeasonalNaive"],
                mode="lines+markers",
                name=f"Naive (MAE={mae_naive:.1f})",
                line={"color": COL_NAIVE, "width": 1.5, "dash": "dot"},
            ),
            row=1,
            col=1,
        )

        # Row 2: Zoom
        fig.add_trace(
            go.Scatter(
                x=test_primary["ds"],
                y=test_primary["y"],
                mode="lines+markers",
                showlegend=False,
                line={"color": COL_ACTUAL, "width": 3},
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_primary["ds"],
                y=test_primary[primary_col],
                mode="lines+markers",
                showlegend=False,
                line=dict(color=COL_SARIMA, width=2, dash="dash"),  # noqa: C408
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_naive["ds"],
                y=test_naive["SeasonalNaive"],
                mode="lines+markers",
                showlegend=False,
                line={"color": COL_NAIVE, "width": 1.5, "dash": "dot"},
            ),
            row=2,
            col=1,
        )

        fig.update_xaxes(title_text="Date", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Sales", row=1, col=1)
        fig.update_yaxes(title_text="Sales", row=2, col=1)
        fig.update_layout(height=800, hovermode="x unified")
        fig.show()

    pred_table = test_primary[["ds", "y"]].rename(columns={"y": "actual"}).copy()

    # primary forecast column depends on model_type (AutoARIMA or Theta)
    pred_table["pred_primary"] = test_primary[primary_col].values
    pred_table["pred_naive"] = test_naive["SeasonalNaive"].values

    # optional but very useful for debugging / filtering
    pred_table["error_primary"] = pred_table["actual"] - pred_table["pred_primary"]
    pred_table["error_naive"] = pred_table["actual"] - pred_table["pred_naive"]

    # add ids for easy grouping in MLflow table UI
    pred_table["pattern"] = pattern
    pred_table["model_type"] = model_type
    pred_table["store_nbr"] = store
    pred_table["item_nbr"] = item
    pred_table["freq"] = freq
    pred_table["season_length"] = season_length
    pred_table["test_weeks"] = test_weeks

    # normalize ds so it logs cleanly
    pred_table["ds"] = pd.to_datetime(pred_table["ds"]).dt.strftime("%Y-%m-%d")

    # log to MLflow (json or parquet; json is fine for small horizons)
    mlflow.log_table(pred_table, "tables/predictions.json")

    params = {
        "pattern": pattern,
        "model": model_name,
        "store": store,
        "item": item,
        "freq": freq,
        "season_length": season_length,
        "test_weeks": test_weeks,
    }

    metrics = {
        "mae_primary": mae_primary,
        "mae_naive": mae_naive,
        "r2_primary": r2_primary,
        "r2_naive": r2_naive,
        "improvement_pct": improvement,
        "n_train": len(train),
        "n_test": len(test),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.log_metrics({"train_size": len(train), "test_size": len(test)})

    results = {**params, **metrics}
    mlflow.end_run()
    return results


print("✅ Baseline function loaded (with debug)")

✅ Baseline function loaded (with debug)


## Run Daily Smooth

In [8]:
daily_smooth_clean = daily_smooth[daily_smooth["date"] < "2016-08-22"].copy()

In [9]:
result_daily_smooth = run_baseline_plotly(
    df=daily_smooth_clean,
    pattern="daily_smooth",
    store=ITEMS_TO_MODEL["daily_smooth"]["store"],
    item=ITEMS_TO_MODEL["daily_smooth"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
    model_type="sarima",
)


🎯 DAILY_SMOOTH | SARIMA
   Using: 'date' (freq=D)
📊 Loaded: 1284 obs | 2013-01-01 00:00:00 to 2016-08-21 00:00:00
   Mean: 10.02, Std: 6.27
  ✅ Filled 45 daily gaps with interpolation
   Final: 1329 observations
✂️  Train: 1301 obs | Test: 28 obs
   Train: 2013-01-01 00:00:00 to 2016-07-24 00:00:00
   Test:  2016-07-25 00:00:00 to 2016-08-21 00:00:00

🤖 Training SARIMA (season=7)...


/var/folders/n0/4xfyfrnd1wj35c4v3fsvg9sh0000gn/T/ipykernel_33967/1701655385.py:80: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="ffill")
/var/folders/n0/4xfyfrnd1wj35c4v3fsvg9sh0000gn/T/ipykernel_33967/1701655385.py:81: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="bfill")


   Forecasting 28 periods...

   DEBUG: Forecasts shape: (28, 3)
   DEBUG: Forecasts columns: ['unique_id', 'ds', 'AutoARIMA']
   DEBUG: Test shape: (28, 3)
   DEBUG: Forecasts reset columns: ['index', 'unique_id', 'ds', 'AutoARIMA']
   DEBUG: First forecast dates: [Timestamp('2016-07-25 00:00:00'), Timestamp('2016-07-26 00:00:00'), Timestamp('2016-07-27 00:00:00')]
   DEBUG: First test dates: [Timestamp('2016-07-25 00:00:00'), Timestamp('2016-07-26 00:00:00'), Timestamp('2016-07-27 00:00:00')]
   DEBUG: After merge, NaN count: 0
   DEBUG: Mask sum: 28 / 28
   ✅ MAE: 4.01 | R²: -0.067

🤖 Training Naive...
   ✅ MAE: 4.61 | R²: -0.597

📊 Improvement: +12.9%


## Run Daily Erratic

In [10]:
result_daily_erratic = run_baseline_plotly(
    df=daily_erratic,
    pattern="daily_erratic",
    store=ITEMS_TO_MODEL["daily_erratic"]["store"],
    item=ITEMS_TO_MODEL["daily_erratic"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
    model_type="sarima",
)


🎯 DAILY_ERRATIC | SARIMA
   Using: 'date' (freq=D)
📊 Loaded: 1580 obs | 2013-01-02 00:00:00 to 2017-08-15 00:00:00
   Mean: 8.73, Std: 7.26
  ✅ Filled 107 daily gaps with interpolation
   Final: 1687 observations
✂️  Train: 1659 obs | Test: 28 obs
   Train: 2013-01-02 00:00:00 to 2017-07-18 00:00:00
   Test:  2017-07-19 00:00:00 to 2017-08-15 00:00:00

🤖 Training SARIMA (season=7)...


/var/folders/n0/4xfyfrnd1wj35c4v3fsvg9sh0000gn/T/ipykernel_33967/1701655385.py:80: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

/var/folders/n0/4xfyfrnd1wj35c4v3fsvg9sh0000gn/T/ipykernel_33967/1701655385.py:81: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



   Forecasting 28 periods...

   DEBUG: Forecasts shape: (28, 3)
   DEBUG: Forecasts columns: ['unique_id', 'ds', 'AutoARIMA']
   DEBUG: Test shape: (28, 3)
   DEBUG: Forecasts reset columns: ['index', 'unique_id', 'ds', 'AutoARIMA']
   DEBUG: First forecast dates: [Timestamp('2017-07-19 00:00:00'), Timestamp('2017-07-20 00:00:00'), Timestamp('2017-07-21 00:00:00')]
   DEBUG: First test dates: [Timestamp('2017-07-19 00:00:00'), Timestamp('2017-07-20 00:00:00'), Timestamp('2017-07-21 00:00:00')]
   DEBUG: After merge, NaN count: 0
   DEBUG: Mask sum: 28 / 28
   ✅ MAE: 2.57 | R²: -0.150

🤖 Training Naive...
   ✅ MAE: 2.82 | R²: -0.401

📊 Improvement: +9.0%


## Run Weekly Smooth (THETA)

In [11]:
result_weekly_smooth = run_baseline_plotly(
    df=weekly_smooth_agg,
    pattern="weekly_smooth",
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
    freq="W",
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
    model_type="theta",
)


🎯 WEEKLY_SMOOTH | THETA
   Using pre-filtered data
📊 Loaded: 147 obs | 2013-12-30 00:00:00 to 2017-08-14 00:00:00
   Mean: 1680.93, Std: 299.56
  ✅ Weekly aggregated data ready (no gap filling needed)
   Final: 147 observations
✂️  Train: 95 obs | Test: 52 obs
   Train: 2013-12-30 00:00:00 to 2016-08-15 00:00:00
   Test:  2016-08-22 00:00:00 to 2017-08-14 00:00:00

🤖 Training Theta (season=52)...
   Forecasting 52 periods...

   DEBUG: Forecasts shape: (52, 3)
   DEBUG: Forecasts columns: ['unique_id', 'ds', 'Theta']
   DEBUG: Test shape: (52, 3)
   DEBUG: Forecasts reset columns: ['index', 'unique_id', 'ds', 'Theta']
   DEBUG: First forecast dates: [Timestamp('2016-08-21 00:00:00'), Timestamp('2016-08-28 00:00:00'), Timestamp('2016-09-04 00:00:00')]
   DEBUG: First test dates: [Timestamp('2016-08-22 00:00:00'), Timestamp('2016-08-29 00:00:00'), Timestamp('2016-09-05 00:00:00')]
   DEBUG: After merge, NaN count: 0
   DEBUG: Mask sum: 52 / 52
   ✅ MAE: 161.74 | R²: -0.114

🤖 Training N

## Run Weekly Erratic (THETA)

In [12]:
result_weekly_erratic = run_baseline_plotly(
    df=weekly_erratic_agg,
    pattern="weekly_erratic",
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
    freq="W",
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
    model_type="theta",
)


🎯 WEEKLY_ERRATIC | THETA
   Using pre-filtered data
📊 Loaded: 179 obs | 2013-11-04 00:00:00 to 2017-07-31 00:00:00
   Mean: 1223.06, Std: 1419.62
  ✅ Weekly aggregated data ready (no gap filling needed)
   Final: 179 observations
✂️  Train: 132 obs | Test: 47 obs
   Train: 2013-11-04 00:00:00 to 2016-08-01 00:00:00
   Test:  2016-08-08 00:00:00 to 2017-07-31 00:00:00

🤖 Training Theta (season=52)...
   Forecasting 47 periods...

   DEBUG: Forecasts shape: (47, 3)
   DEBUG: Forecasts columns: ['unique_id', 'ds', 'Theta']
   DEBUG: Test shape: (47, 3)
   DEBUG: Forecasts reset columns: ['index', 'unique_id', 'ds', 'Theta']
   DEBUG: First forecast dates: [Timestamp('2016-08-07 00:00:00'), Timestamp('2016-08-14 00:00:00'), Timestamp('2016-08-21 00:00:00')]
   DEBUG: First test dates: [Timestamp('2016-08-08 00:00:00'), Timestamp('2016-08-15 00:00:00'), Timestamp('2016-08-22 00:00:00')]
   DEBUG: After merge, NaN count: 0
   DEBUG: Mask sum: 47 / 47
   ✅ MAE: 546.72 | R²: -0.171

🤖 Trainin

## Summary

In [1]:
# SUMMARY

results = [
    result_daily_smooth,
    result_daily_erratic,
    result_weekly_smooth,
    result_weekly_erratic,
]

summary_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("📊 BASELINE SUMMARY")
print("=" * 70)
print(
    summary_df[
        [
            "pattern",
            "model",
            "test_weeks",
            "n_test",
            "mae_primary",
            "mae_naive",
            "improvement_pct",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("CONFIGURATION:")
print("=" * 70)
print("  Daily:  SARIMA, Linear Interpolation")
print("  Weekly: THETA, Smart Gap Filling (<5%: Interpolate | ≥5%: Drop)")

# Bar Chart
fig = go.Figure()
x = summary_df["pattern"]
colors = ["red" if "daily" in p else "purple" for p in x]

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_primary"],
        name="Primary Model",
        marker_color=colors,
        opacity=0.8,
        text=summary_df["mae_primary"].round(2),
        textposition="outside",
        customdata=summary_df["model"],
        hovertemplate="%{x}<br>%{customdata} MAE: %{y:.2f}<extra></extra>",
    )
)

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_naive"],
        name="Naive",
        marker_color="orange",
        opacity=0.6,
        text=summary_df["mae_naive"].round(2),
        textposition="outside",
    )
)

for i, row in summary_df.iterrows():
    fig.add_annotation(
        x=i,
        y=max(row["mae_primary"], row["mae_naive"]) + 2,
        text=f"{row['improvement_pct']:+.1f}%",
        showarrow=False,
        font={
            "size": 14,
            "color": "green" if row["improvement_pct"] > 0 else "red",
            "family": "Arial Black",
        },
    )

fig.update_layout(
    title="Baseline Model Comparison<br><sub>Daily: SARIMA | Weekly: THETA</sub>",
    xaxis_title="Pattern",
    yaxis_title="MAE",
    barmode="group",
    template=TEMPLATE,
    height=500,
)
fig.show()

# Save
summary_df.to_csv("baseline_summary_final.csv", index=False)
print("\n✅ Saved: baseline_summary_final.csv")

NameError: name 'result_daily_smooth' is not defined